# Agentic Shield: Preventing Hallucinations in LangGraph Agents

**Goal:** Learn how to use the Hallucination Shield to validate agent actions and prevent error propagation.

In this tutorial, you'll learn:
1. How hallucinations compound in multi-step agentic workflows
2. Setting up the Hallucination Shield
3. Building a simple LangGraph agent
4. Validating agent actions before execution
5. Handling rejected actions gracefully

---

## Why Shield Agentic Workflows?

In multi-step agent systems, a single hallucination can cascade:
- **Step 1:** Agent misinterprets context and generates incorrect tool call
- **Step 2:** Incorrect tool output becomes new context
- **Step 3:** Agent builds upon false information
- **Result:** Compounding errors leading to completely invalid outcomes

The Hallucination Shield acts as a validation checkpoint, allowing the agent to propose actions while the jury validates them against source context.

---

## Setup

In [ ]:
# API Configuration
import os

GROQ_API_KEY = "gsk_YOUR_API_KEY_HERE"

# Or from environment
# GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [3]:
# Imports
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, List
import operator

from llm_jury.core.evaluator import JuryEvaluator
from llm_jury.judges.llm_judge import LLMJudge
from llm_jury.metrics.predefined import GroundednessMetric
from llm_jury.strategies.consensus import MajorityVoting
from llm_jury.tools.shield import HallucinationShield

print("All imports successful!")

All imports successful!


## Configure the Jury Panel

We'll use three different Groq models as judges to form a diverse panel.

In [4]:
# Initialize three different models as judges
judge1 = LLMJudge(
    model=ChatGroq(model="llama-3.3-70b-versatile", api_key=GROQ_API_KEY, temperature=0.1),
    name="llama-3.3-70b"
)

judge2 = LLMJudge(
    model=ChatGroq(model="llama-3.1-8b-instant", api_key=GROQ_API_KEY, temperature=0.1),
    name="llama-3.1-8b"
)

judge3 = LLMJudge(
    model=ChatGroq(model="openai/gpt-oss-120b", api_key=GROQ_API_KEY, temperature=0.1),
    name="gpt-oss-120b"
)

# Create the jury evaluator
jury = JuryEvaluator(
    judges=[judge1, judge2, judge3],
    strategy=MajorityVoting()
)

# Initialize the shield
shield = HallucinationShield(jury_evaluator=jury)

print("Hallucination Shield initialized with 3-judge panel")
print(f"Judges: {[j.name for j in jury.judges]}")

Hallucination Shield initialized with 3-judge panel
Judges: ['llama-3.3-70b', 'llama-3.1-8b', 'gpt-oss-120b']


## Example 1: Simple Research Agent with Shield

Let's build a research agent that analyzes documents and extracts key information. The shield validates each extraction step.

In [5]:
# Define the agent state
class AgentState(TypedDict):
    source_document: str
    query: str
    proposed_answer: str
    validation_result: dict
    final_answer: str
    messages: Annotated[List[str], operator.add]

In [6]:
# Initialize the agent model
agent_model = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    temperature=0.3
)

def generate_answer(state: AgentState) -> AgentState:
    """
    Agent generates a proposed answer based on the source document.
    """
    prompt = f"""
You are a research assistant. Based on the following document, answer the user's question.

Document:
{state['source_document']}

Question: {state['query']}

Provide a concise answer based ONLY on the information in the document.
"""
    
    response = agent_model.invoke([HumanMessage(content=prompt)])
    proposed = response.content
    
    return {
        **state,
        "proposed_answer": proposed,
        "messages": [f"[Agent] Generated proposed answer"]
    }

def validate_with_shield(state: AgentState) -> AgentState:
    """
    Use the Hallucination Shield to validate the proposed answer.
    """
    validation = shield.validate_step(
        context_text=state['source_document'],
        proposed_action=state['proposed_answer'],
        metric=GroundednessMetric()
    )
    
    return {
        **state,
        "validation_result": {
            "is_valid": validation.is_valid,
            "confidence": validation.confidence,
            "reasoning": validation.consensus_reasoning
        },
        "messages": [
            f"[Shield] Validation: {'PASSED' if validation.is_valid else 'REJECTED'} "
            f"(confidence: {validation.confidence:.2f})"
        ]
    }

def finalize_answer(state: AgentState) -> AgentState:
    """
    Decide on final answer based on validation result.
    """
    if state['validation_result']['is_valid']:
        final = state['proposed_answer']
        message = "[Agent] Answer approved by shield"
    else:
        # In a real system, the agent would revise here
        final = "Unable to provide a grounded answer. The proposed response was not supported by the source document."
        message = "[Agent] Answer rejected - cannot provide grounded response"
    
    return {
        **state,
        "final_answer": final,
        "messages": [message]
    }

print("Agent node functions defined")

Agent node functions defined


In [7]:
# Build the LangGraph workflow
workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("generate", generate_answer)
workflow.add_node("validate", validate_with_shield)
workflow.add_node("finalize", finalize_answer)

# Define the flow
workflow.set_entry_point("generate")
workflow.add_edge("generate", "validate")
workflow.add_edge("validate", "finalize")
workflow.add_edge("finalize", END)

# Compile the graph
research_agent = workflow.compile()

print("Research agent graph compiled successfully")

Research agent graph compiled successfully


## Test Case 1: Valid Grounded Answer

Let's test with a query that has clear support in the document.

In [8]:
# Source document
source_doc = """
Anthropic's Claude 3.5 Sonnet was released in June 2024. It features significant improvements 
in coding, visual analysis, and mathematical reasoning compared to previous versions. The model 
has a 200K token context window and operates at twice the speed of Claude 3 Opus while maintaining 
comparable intelligence levels. Claude 3.5 Sonnet is available through the Claude API, Amazon 
Bedrock, and Google Cloud's Vertex AI.
"""

# Query with clear answer in the document
query_1 = "What is the context window size of Claude 3.5 Sonnet?"

# Run the agent
result_1 = research_agent.invoke({
    "source_document": source_doc,
    "query": query_1,
    "proposed_answer": "",
    "validation_result": {},
    "final_answer": "",
    "messages": []
})

print("="*80)
print("TEST CASE 1: Valid Grounded Answer")
print("="*80)
print(f"\nQuery: {query_1}")
print(f"\nProposed Answer:\n{result_1['proposed_answer']}")
print(f"\nValidation Status: {'PASSED' if result_1['validation_result']['is_valid'] else 'REJECTED'}")
print(f"Confidence: {result_1['validation_result']['confidence']:.2f}")
print(f"\nFinal Answer:\n{result_1['final_answer']}")
print(f"\nExecution Log:")
for msg in result_1['messages']:
    print(f"  {msg}")

TEST CASE 1: Valid Grounded Answer

Query: What is the context window size of Claude 3.5 Sonnet?

Proposed Answer:
The context window size of Claude 3.5 Sonnet is 200K tokens.

Validation Status: PASSED
Confidence: 1.00

Final Answer:
The context window size of Claude 3.5 Sonnet is 200K tokens.

Execution Log:
  [Agent] Generated proposed answer
  [Shield] Validation: PASSED (confidence: 1.00)
  [Agent] Answer approved by shield


## Test Case 2: Hallucinated Answer

Let's test with a query that might tempt the agent to hallucinate information not in the document.

In [9]:
# Query that's not answered in the document
query_2 = "What is Claude 3.5 Sonnet's pricing per million tokens?"

# Run the agent
result_2 = research_agent.invoke({
    "source_document": source_doc,
    "query": query_2,
    "proposed_answer": "",
    "validation_result": {},
    "final_answer": "",
    "messages": []
})

print("="*80)
print("TEST CASE 2: Information Not in Document")
print("="*80)
print(f"\nQuery: {query_2}")
print(f"\nProposed Answer:\n{result_2['proposed_answer']}")
print(f"\nValidation Status: {'PASSED' if result_2['validation_result']['is_valid'] else 'REJECTED'}")
print(f"Confidence: {result_2['validation_result']['confidence']:.2f}")
print(f"\nJury Reasoning:\n{result_2['validation_result']['reasoning'][:300]}...")
print(f"\nFinal Answer:\n{result_2['final_answer']}")
print(f"\nExecution Log:")
for msg in result_2['messages']:
    print(f"  {msg}")

TEST CASE 2: Information Not in Document

Query: What is Claude 3.5 Sonnet's pricing per million tokens?

Proposed Answer:
There is no information about Claude 3.5 Sonnet's pricing per million tokens in the document.

Validation Status: PASSED
Confidence: 1.00

Jury Reasoning:
The output statement is a direct conclusion based on the information provided in the source text. The source text does not mention pricing per million tokens for Claude 3.5 Sonnet, making the output statement accurate and fully supported by the source text. | The output statement "There is no inform...

Final Answer:
There is no information about Claude 3.5 Sonnet's pricing per million tokens in the document.

Execution Log:
  [Agent] Generated proposed answer
  [Shield] Validation: PASSED (confidence: 1.00)
  [Agent] Answer approved by shield


## Example 2: Multi-Step Analysis Agent

Now let's build a more complex agent that performs multi-step analysis with shield validation at each step.

In [10]:
# Enhanced state for multi-step analysis
class AnalysisState(TypedDict):
    source_document: str
    analysis_task: str
    step_1_result: str
    step_1_valid: bool
    step_2_result: str
    step_2_valid: bool
    final_synthesis: str
    messages: Annotated[List[str], operator.add]

In [11]:
# Analysis agent model
analysis_model = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    temperature=0.2
)

def extract_key_facts(state: AnalysisState) -> AnalysisState:
    """
    Step 1: Extract key facts from the document.
    """
    prompt = f"""
Extract the 3 most important facts from this document:

{state['source_document']}

List them as bullet points.
"""
    
    response = analysis_model.invoke([HumanMessage(content=prompt)])
    extraction = response.content
    
    # Validate with shield
    validation = shield.validate_step(
        context_text=state['source_document'],
        proposed_action=extraction,
        metric=GroundednessMetric()
    )
    
    return {
        **state,
        "step_1_result": extraction,
        "step_1_valid": validation.is_valid,
        "messages": [
            f"[Step 1] Extracted key facts",
            f"[Shield] Validation: {'PASSED' if validation.is_valid else 'REJECTED'} "
            f"(confidence: {validation.confidence:.2f})"
        ]
    }

def analyze_implications(state: AnalysisState) -> AnalysisState:
    """
    Step 2: Analyze implications based on extracted facts (only if step 1 passed).
    """
    if not state['step_1_valid']:
        return {
            **state,
            "step_2_result": "Skipped due to step 1 validation failure",
            "step_2_valid": False,
            "messages": ["[Step 2] Skipped - previous step invalid"]
        }
    
    prompt = f"""
Based on these facts from the document:
{state['step_1_result']}

Original document:
{state['source_document']}

Provide a brief analysis of what these facts suggest about the topic. 
Base your analysis ONLY on explicitly stated information.
"""
    
    response = analysis_model.invoke([HumanMessage(content=prompt)])
    analysis = response.content
    
    # Validate with shield
    validation = shield.validate_step(
        context_text=state['source_document'],
        proposed_action=analysis,
        metric=GroundednessMetric()
    )
    
    return {
        **state,
        "step_2_result": analysis,
        "step_2_valid": validation.is_valid,
        "messages": [
            f"[Step 2] Generated analysis",
            f"[Shield] Validation: {'PASSED' if validation.is_valid else 'REJECTED'} "
            f"(confidence: {validation.confidence:.2f})"
        ]
    }

def synthesize_results(state: AnalysisState) -> AnalysisState:
    """
    Final step: Synthesize results if all steps passed.
    """
    if state['step_1_valid'] and state['step_2_valid']:
        synthesis = f"""Analysis Complete:

Key Facts:
{state['step_1_result']}

Analysis:
{state['step_2_result']}

All steps validated by hallucination shield."""
        message = "[Synthesis] All steps valid - analysis complete"
    else:
        synthesis = "Analysis incomplete: One or more steps failed validation."
        message = "[Synthesis] Validation failures prevented complete analysis"
    
    return {
        **state,
        "final_synthesis": synthesis,
        "messages": [message]
    }

print("Multi-step analysis functions defined")

Multi-step analysis functions defined


In [12]:
# Build the multi-step analysis workflow
analysis_workflow = StateGraph(AnalysisState)

# Add nodes
analysis_workflow.add_node("extract", extract_key_facts)
analysis_workflow.add_node("analyze", analyze_implications)
analysis_workflow.add_node("synthesize", synthesize_results)

# Define the flow
analysis_workflow.set_entry_point("extract")
analysis_workflow.add_edge("extract", "analyze")
analysis_workflow.add_edge("analyze", "synthesize")
analysis_workflow.add_edge("synthesize", END)

# Compile
analysis_agent = analysis_workflow.compile()

print("Multi-step analysis agent compiled successfully")

Multi-step analysis agent compiled successfully


## Test the Multi-Step Agent

In [13]:
# Technical document for analysis
tech_doc = """
LangGraph is a framework for building stateful, multi-actor applications with Large Language Models. 
It extends LangChain by adding cyclic computational graphs and built-in persistence. Key features include:
1) State management with automatic checkpointing
2) Support for human-in-the-loop workflows
3) Streaming support for real-time outputs
4) Built-in graph visualization tools

LangGraph is particularly well-suited for building autonomous agents, complex RAG systems, and 
multi-step reasoning workflows. It uses a StateGraph class to define nodes and edges, where each 
node represents a function that transforms the state.
"""

# Run multi-step analysis
analysis_result = analysis_agent.invoke({
    "source_document": tech_doc,
    "analysis_task": "Analyze LangGraph capabilities",
    "step_1_result": "",
    "step_1_valid": False,
    "step_2_result": "",
    "step_2_valid": False,
    "final_synthesis": "",
    "messages": []
})

print("="*80)
print("MULTI-STEP ANALYSIS WITH SHIELD VALIDATION")
print("="*80)
print(f"\nExecution Log:")
for msg in analysis_result['messages']:
    print(f"  {msg}")
print(f"\n{'='*80}")
print(f"FINAL SYNTHESIS:")
print(f"{'='*80}")
print(analysis_result['final_synthesis'])

MULTI-STEP ANALYSIS WITH SHIELD VALIDATION

Execution Log:
  [Step 1] Extracted key facts
  [Shield] Validation: PASSED (confidence: 0.67)
  [Step 2] Generated analysis
  [Shield] Validation: PASSED (confidence: 0.67)
  [Synthesis] All steps valid - analysis complete

FINAL SYNTHESIS:
Analysis Complete:

Key Facts:
Here are the 3 most important facts about LangGraph:

* LangGraph is a framework for building stateful, multi-actor applications with Large Language Models, extending LangChain with additional features.
* Key features of LangGraph include state management, human-in-the-loop workflows, and streaming support for real-time outputs.
* LangGraph is well-suited for building complex applications such as autonomous agents, RAG systems, and multi-step reasoning workflows, using a StateGraph class to define nodes and edges.

Analysis:
Based on the provided facts, LangGraph appears to be a powerful framework for building complex applications that utilize Large Language Models. The key 

## Understanding Shield Behavior

Let's examine what the shield caught in our validation steps.

In [14]:
# Direct shield validation example
good_statement = "LangGraph has built-in persistence and checkpointing capabilities."
bad_statement = "LangGraph was released in 2022 and has over 1 million users worldwide."

print("="*80)
print("DIRECT SHIELD VALIDATION EXAMPLES")
print("="*80)

# Test grounded statement
result_good = shield.validate_step(
    context_text=tech_doc,
    proposed_action=good_statement,
    metric=GroundednessMetric()
)

print(f"\nStatement: {good_statement}")
print(f"Validation: {'PASSED' if result_good.is_valid else 'REJECTED'}")
print(f"Confidence: {result_good.confidence:.2f}")
print(f"Reasoning: {result_good.consensus_reasoning[:200]}...")

print(f"\n{'-'*80}\n")

# Test ungrounded statement
result_bad = shield.validate_step(
    context_text=tech_doc,
    proposed_action=bad_statement,
    metric=GroundednessMetric()
)

print(f"Statement: {bad_statement}")
print(f"Validation: {'PASSED' if result_bad.is_valid else 'REJECTED'}")
print(f"Confidence: {result_bad.confidence:.2f}")
print(f"Reasoning: {result_bad.consensus_reasoning[:200]}...")

DIRECT SHIELD VALIDATION EXAMPLES

Statement: LangGraph has built-in persistence and checkpointing capabilities.
Validation: PASSED
Confidence: 1.00
Reasoning: The output is fully supported by the source text. The source text explicitly mentions "built-in persistence" and "automatic checkpointing" as key features of LangGraph. The output is a concise rephras...

--------------------------------------------------------------------------------

Statement: LangGraph was released in 2022 and has over 1 million users worldwide.
Validation: REJECTED
Confidence: 0.67
Reasoning: The output contains significant information not present in the source. The source text does not mention the release year of LangGraph or the number of users worldwide. It only provides a description o...


## Key Takeaways

### Benefits of the Hallucination Shield

**1. Prevents Error Propagation**
   - Validates each step before proceeding
   - Stops hallucinations from becoming context for future steps
   - Maintains grounding throughout multi-step workflows

**2. Provides Transparency**
   - Clear validation results at each checkpoint
   - Detailed reasoning from multiple judges
   - Confidence scores for decision-making

**3. Enables Graceful Degradation**
   - Agent can adapt when validation fails
   - Can skip dependent steps if validation fails
   - User gets informed about limitations

**4. Improves Reliability**
   - Multiple judge consensus reduces false positives
   - Systematic validation reduces manual oversight
   - Audit trail for debugging and improvement

### Best Practices

1. Validate at critical decision points
2. Use appropriate metrics for each step type
3. Configure confidence thresholds based on risk
4. Log validation results for analysis
5. Provide recovery guidance when validation fails

### When to Use the Shield

- Multi-step agentic workflows
- RAG systems with complex reasoning
- High-stakes decision-making agents
- Any system where error propagation is costly
- Production deployments requiring auditability

---

## Next Steps

- Integrate shield into your existing agents
- Experiment with different validation thresholds
- Add custom metrics for domain-specific validation
- Build feedback loops for agent improvement
- Monitor validation patterns over time

---

**The Hallucination Shield: Your Agent's Safety Net!**